# Data Prep for EV Energy Consumption Dataset

## Step 1: Initial Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

In [ ]:
data = pd.read_csv('data/raw/5-EV_Energy_Consumption_Dataset.csv')
print("Shape:", data.shape)
data.head()

In [ ]:
print("--- Data Types ---")
print(data.dtypes)
print("\n--- Basic Statistics ---")
data.describe()

## Step 2: Handling Missing Values

In [ ]:
# Identify missing values per column
print("Missing values per column:")
print(data.isna().sum())
print(f"\nTotal missing: {data.isna().sum().sum()}")

In [ ]:
# Drop columns where more than 50% of values are missing
threshold = len(data) * 0.5
data_clean = data.dropna(thresh=threshold, axis=1)

# Drop non-informative columns: Vehicle_ID (identifier) and Timestamp (not used as a feature)
data_clean = data_clean.drop(columns=['Vehicle_ID', 'Timestamp'], errors='ignore')

# Impute remaining numeric columns with median
numeric_cols = data_clean.select_dtypes(include=np.number).columns
data_clean[numeric_cols] = data_clean[numeric_cols].fillna(data_clean[numeric_cols].median())

print("Missing values after imputation:")
print(data_clean.isna().sum())
print(f"\nShape after handling missing values: {data_clean.shape}")

## Step 3: Dealing with Duplicates

In [ ]:
print(f"Duplicate rows found: {data_clean.duplicated().sum()}")
data_clean = data_clean.drop_duplicates()
print(f"Shape after removing duplicates: {data_clean.shape}")

## Step 4: Handling Categorical Data

`Driving_Mode`, `Road_Type`, `Traffic_Condition`, and `Weather_Condition` are stored as integers but represent nominal categories — they need to be one-hot encoded.

In [ ]:
# Cast integer-coded nominal columns to category type
nominal_cols = ['Driving_Mode', 'Road_Type', 'Traffic_Condition', 'Weather_Condition']
for col in nominal_cols:
    data_clean[col] = data_clean[col].astype('category')

print("Category levels:")
for col in nominal_cols:
    print(f"  {col}: {sorted(data_clean[col].cat.categories.tolist())}")

# One-hot encode all nominal columns
data_encoded = pd.get_dummies(data_clean, columns=nominal_cols, drop_first=False)
print(f"\nShape after one-hot encoding: {data_encoded.shape}")
data_encoded.head()

## Step 5: Data Manipulation & Outlier Management

In [ ]:
# --- Discretization: bin Battery_State_% into meaningful charge groups ---
data_encoded['Charge_Level'] = pd.cut(
    data_encoded['Battery_State_%'],
    bins=[0, 20, 40, 60, 80, 100],
    labels=['Critical', 'Low', 'Moderate', 'Good', 'Full'],
    include_lowest=True
)
print("Charge Level distribution:")
print(data_encoded['Charge_Level'].value_counts().sort_index())

# --- Outlier Capping (±3 standard deviations) ---
# Only cap continuous numeric features; skip binary dummy columns
numeric_features = data_encoded.select_dtypes(include=np.number).columns.tolist()
exclude_cols = [c for c in numeric_features if data_encoded[c].nunique() <= 2]
cap_cols = [c for c in numeric_features if c not in exclude_cols]

outlier_counts = {}
for col in cap_cols:
    mean, std = data_encoded[col].mean(), data_encoded[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std
    n_outliers = ((data_encoded[col] < lower) | (data_encoded[col] > upper)).sum()
    if n_outliers > 0:
        outlier_counts[col] = n_outliers
    data_encoded[col] = data_encoded[col].clip(lower, upper)

print(f"\nOutliers capped per column: {outlier_counts}")
print(f"Shape after outlier treatment: {data_encoded.shape}")

## Step 6: Data Partitioning (Train/Test Split)

In [ ]:
# Drop derived/categorical columns not needed for modelling
df_model = data_encoded.drop(columns=['Charge_Level'])

X = df_model.drop(columns=['Energy_Consumption_kWh'])
y = df_model['Energy_Consumption_kWh']

# Regression target — 80/20 split (no stratify needed for continuous targets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set:  {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:      {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTarget stats (train):")
print(y_train.describe())

## Step 7: Feature Scaling

In [ ]:
# Scale only continuous numeric features (exclude binary dummy columns)
scale_cols = [c for c in X_train.select_dtypes(include=np.number).columns
              if X_train[c].nunique() > 2]

# --- Standardization (preferred: mean=0, std=1) ---
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

# Fit ONLY on training data, then transform both sets to prevent data leakage
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

print("Standardization applied to:", scale_cols)
print(f"\nTrain feature means (should be ~0):\n{X_train_scaled[scale_cols].mean().round(4)}")
print(f"\nTrain feature std (should be ~1):\n{X_train_scaled[scale_cols].std().round(4)}")

# --- MinMax Normalization (for reference) ---
minmax = MinMaxScaler()
X_train_norm = X_train.copy()
X_test_norm  = X_test.copy()
X_train_norm[scale_cols] = minmax.fit_transform(X_train[scale_cols])
X_test_norm[scale_cols]  = minmax.transform(X_test[scale_cols])
print(f"\nMinMax range check (train) — min:\n{X_train_norm[scale_cols].min().round(4)}")
print(f"MinMax range check (train) — max:\n{X_train_norm[scale_cols].max().round(4)}")

## Step 8: Visual Exploration

In [ ]:
# --- Univariate: Distributions of key numeric features ---
cont_features = ['Energy_Consumption_kWh', 'Speed_kmh', 'Battery_State_%',
                 'Battery_Temperature_C', 'Vehicle_Weight_kg',
                 'Distance_Travelled_km', 'Slope_%', 'Tire_Pressure_psi']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cont_features):
    sns.histplot(data_clean[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)

for j in range(len(cont_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Univariate Distributions of Key Numeric Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Multivariate: Average Energy Consumption by Driving Mode and Road Type ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot: Average Energy Consumption by Driving Mode
driving_energy = data_clean.groupby('Driving_Mode')['Energy_Consumption_kWh'].mean().sort_values(ascending=False)
driving_energy.plot.bar(ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Avg Energy Consumption by Driving Mode')
axes[0].set_xlabel('Driving Mode')
axes[0].set_ylabel('Avg Energy Consumption (kWh)')
axes[0].tick_params(axis='x', rotation=0)

# Bar plot: Average Energy Consumption by Road Type
road_energy = data_clean.groupby('Road_Type')['Energy_Consumption_kWh'].mean().sort_values(ascending=False)
road_energy.plot.bar(ax=axes[1], color='mediumseagreen', edgecolor='black')
axes[1].set_title('Avg Energy Consumption by Road Type')
axes[1].set_xlabel('Road Type')
axes[1].set_ylabel('Avg Energy Consumption (kWh)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap ---
numeric_cols_plot = ['Speed_kmh', 'Acceleration_ms2', 'Battery_State_%', 'Battery_Voltage_V',
                     'Battery_Temperature_C', 'Slope_%', 'Temperature_C', 'Humidity_%',
                     'Wind_Speed_ms', 'Tire_Pressure_psi', 'Vehicle_Weight_kg',
                     'Distance_Travelled_km', 'Energy_Consumption_kWh']

corr = data_clean[numeric_cols_plot].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()